# Chess Personas — independent player models
Train separate Hikaru and Caruana models from a frozen Maia-3 backbone.
Each model learns only from its target player's positions and has its own
move scorer, adapter, history/context layers, and timing head.

Select **Runtime → Change runtime type → GPU**. This notebook starts with three
months of public games and a position cap. It does not launch automatically.
Caruana remains a provisional second persona until clock coverage is checked.

For a private repository, download its ZIP from GitHub's **Code → Download ZIP**
menu, then upload it below. Credentials are not embedded in this notebook.


In [ ]:
from pathlib import Path
import io, os, subprocess, sys, zipfile
from google.colab import files

workspace = Path('/content/chess-personas-work')
workspace.mkdir(exist_ok=True)
zip_path = Path('/content/Maia-Chess-starter.zip')
uploaded = {zip_path.name: zip_path.read_bytes()} if zip_path.exists() else files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one repository ZIP'
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(workspace)
projects = [p.parent for p in workspace.rglob('pyproject.toml')
            if (p.parent / 'requirements-maia.txt').exists()]
assert len(projects) == 1, 'Expected one Chess Personas repository'
os.chdir(projects[0])
print('Repository:', Path.cwd())

def run(*args):
    command = [sys.executable, '-u', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

run('-m', 'pip', 'install', '-e', '.[train,dev]')
run('-m', 'pip', 'install', '-r', 'requirements-maia.txt')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to GPU for training')
run('-m', 'pytest', '-q')

## Download and inspect data
Public archive requests are sequential and cached. Increase the month count only
after the first run works. A full-history request is `--max-months 0`.
If preparing a second time, choose a new output directory to preserve old splits.

In [ ]:
run('-m', 'chess_personas.cli', 'download', 'hikaru', 'fabianocaruana', '--max-months', '3')
run('-m', 'chess_personas.cli', 'prepare', 'data/raw/*/*.json', '--output', 'data/processed')
import json
audit = json.loads(Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2))

## Train one model per player
Both players must have training and validation positions and useful timing
coverage. Each run filters the prepared files to one player, starts from base
Maia with fresh trainable layers, and selects its own best checkpoint.

This example explicitly samples 10,000 training positions per player; validation is uncapped. Each model runs for 10 epochs
with batch size 32; reduce the batch size to 8 if GPU memory is insufficient.
Use a new `RUN_ROOT` for another experiment. Models do not resume previous runs.


In [ ]:
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
RUN_ROOT = Path('runs/single-player')
PLAYER_RUNS = {player: RUN_ROOT / player for player in ('hikaru', 'fabianocaruana')}
assert not RUN_ROOT.exists(), 'Choose a new RUN_ROOT to preserve previous models'
for player, run_dir in PLAYER_RUNS.items():
    print('Training independent model:', player, flush=True)
    run('-m', 'chess_personas.cli', 'train', '--player', player,
        '--output', run_dir, '--epochs', '10', '--batch-size', '32',
        '--max-per-player', '10000', '--seed', '42')


## Evaluate each checkpoint separately
Evaluation filters the held-out file to the checkpoint's sole player. Compare
move accuracy, move NLL, timing NLL, and timing error on that player's games.
The example position is synthetic. Its player is inferred from each checkpoint.


In [ ]:
for player, run_dir in PLAYER_RUNS.items():
    config = json.loads((run_dir / 'config.json').read_text())
    assert config['players'] == [player]
    print('Evaluating independent model:', player, flush=True)
    run('-m', 'chess_personas.cli', 'evaluate', run_dir / 'best.pt',
        '--output', run_dir / 'test-metrics.json')
    run('-m', 'chess_personas.cli', 'predict', run_dir / 'best.pt',
        '--record', 'examples/position.json')


## Save both models before the session ends
The archive contains a `hikaru/` folder and a `fabianocaruana/` folder. Each has
its own `best.pt`, configuration, validation history, and test metrics.
Each checkpoint embeds the frozen backbone and its independently trained layers.


In [ ]:
import shutil
archive_path = shutil.make_archive('/content/chess-personas-separate-players', 'zip', RUN_ROOT)
files.download(archive_path)
